# DS-02 위험 변화 분석

이 노트북은 EPSS 시계열과 KEV 현재 등재 특성을 분석해 재검토 후보 신호를 확인합니다.

- 범위: EPSS 윈도우별 후보 수, KEV 등재 CVE와 EPSS/GHSA proxy overlap, 입력용 후보 신호 확인.
- 이 노트북은 Lakehouse 테이블에 쓰지 않고 temp view만 생성합니다.

## 1. 실행 파라미터

In [ ]:
from pyspark.sql import functions as F

WINDOWS = {
    "d1": 1,
    "d7": 7,
    "d30": 30,
    "d90": 90,
}

EPSILON = 1e-12

print({"windows": WINDOWS})

## 2. EPSS 원본 로드 및 품질 확인

In [ ]:
epss_lines = (
    spark.read
    .text("Files/bronze/epss/*.csv.gz")
    .withColumn("source_file", F.input_file_name())
)

epss_parsed = (
    epss_lines
    .filter(~F.col("value").startswith("#model_version"))
    .filter(F.col("value") != "cve,epss,percentile")
    .withColumn("parts", F.split("value", ","))
    .select(
        F.to_date(
            F.regexp_extract("source_file", r"epss_scores-(\d{4}-\d{2}-\d{2})\.csv\.gz", 1)
        ).alias("score_date"),
        F.col("parts").getItem(0).alias("cve_id"),
        F.col("parts").getItem(1).cast("double").alias("epss"),
        F.col("parts").getItem(2).cast("double").alias("percentile"),
        F.col("source_file")
    )
)

epss_dq_summary = epss_parsed.agg(
    F.count("*").alias("parsed_rows"),
    F.sum(F.when(F.col("score_date").isNull(), 1).otherwise(0)).alias("missing_score_date_rows"),
    F.sum(F.when(~F.col("cve_id").startswith("CVE-"), 1).otherwise(0)).alias("invalid_cve_rows"),
    F.sum(F.when(F.col("epss").isNull(), 1).otherwise(0)).alias("missing_epss_rows"),
    F.sum(F.when(F.col("percentile").isNull(), 1).otherwise(0)).alias("missing_percentile_rows"),
)

epss = (
    epss_parsed
    .filter(F.col("score_date").isNotNull())
    .filter(F.col("cve_id").startswith("CVE-"))
    .filter(F.col("epss").isNotNull())
    .filter(F.col("percentile").isNotNull())
    .dropDuplicates(["score_date", "cve_id"])
)

display(epss_dq_summary)
display(
    epss.groupBy("score_date")
    .agg(F.countDistinct("cve_id").alias("cve_count"))
    .orderBy("score_date")
)
epss.printSchema()

## 3. EPSS 윈도우별 feature 구성

In [ ]:
detection_date = epss.agg(F.max("score_date").alias("detection_date")).collect()[0]["detection_date"]
print("detection_date =", detection_date)

current = (
    epss
    .filter(F.col("score_date") == F.lit(detection_date))
    .select(
        "cve_id",
        F.col("epss").alias("epss_current"),
        F.col("percentile").alias("percentile_current"),
    )
)

epss_features = current

for label, days in WINDOWS.items():
    baseline = (
        epss
        .filter(F.col("score_date") == F.date_sub(F.lit(detection_date), days))
        .select(
            "cve_id",
            F.col("epss").alias(f"epss_{label}"),
            F.col("percentile").alias(f"percentile_{label}"),
        )
    )

    epss_features = (
        epss_features
        .join(baseline, on="cve_id", how="left")
        .withColumn(f"epss_delta_{label}", F.col("epss_current") - F.col(f"epss_{label}"))
        .withColumn(
            f"epss_ratio_{label}",
            F.when(F.col(f"epss_{label}") > 0, F.col("epss_current") / F.col(f"epss_{label}"))
        )
        .withColumn(f"percentile_delta_{label}", F.col("percentile_current") - F.col(f"percentile_{label}"))
    )

epss_features.createOrReplaceTempView("ds02_epss_features")

display(epss_features.limit(20))

## 4. EPSS 고위험 진입 후보 수

In [ ]:
features_with_r_epss_01 = epss_features

for label in WINDOWS.keys():
    features_with_r_epss_01 = (
        features_with_r_epss_01
        .withColumn(
            f"r_epss_01a_top_10_entry_{label}",
            (F.col(f"percentile_{label}") < 0.90) & (F.col("percentile_current") >= 0.90),
        )
        .withColumn(
            f"r_epss_01b_top_5_entry_{label}",
            (F.col(f"percentile_{label}") < 0.90) & (F.col("percentile_current") >= 0.95),
        )
        .withColumn(
            f"r_epss_01c_top_1_entry_{label}",
            (F.col(f"percentile_{label}") < 0.90) & (F.col("percentile_current") >= 0.99),
        )
        .withColumn(
            f"r_epss_01d_high_persistent_{label}",
            (F.col(f"percentile_{label}") >= 0.95) & (F.col("percentile_current") >= 0.95),
        )
        .withColumn(
            f"r_epss_01e_high_entry_then_persistent_{label}",
            (F.col(f"percentile_{label}") < 0.90)
            & (F.col("percentile_current") >= 0.95)
            & (F.col("percentile_d7") >= 0.95),
        )
    )

rule_cols = [column for column in features_with_r_epss_01.columns if column.startswith("r_epss_01")]
total_cve_count = features_with_r_epss_01.select("cve_id").distinct().count()

summary_rows = []
for rule_col in rule_cols:
    hit_count = features_with_r_epss_01.filter(F.col(rule_col) == True).count()
    hit_rate_pct = round((hit_count / total_cve_count) * 100, 4)
    summary_rows.append((rule_col, hit_count, hit_rate_pct))

r_epss_01_summary = spark.createDataFrame(summary_rows, ["rule_id", "hit_count", "hit_rate_pct"])
r_epss_01_summary.createOrReplaceTempView("ds02_r_epss_01_summary")

display(r_epss_01_summary.orderBy(F.desc("hit_count")))

## 5. EPSS 급상승 후보 수

In [ ]:
features_with_r_epss_02 = epss_features

for label in WINDOWS.keys():
    features_with_r_epss_02 = (
        features_with_r_epss_02
        .withColumn(
            f"r_epss_02a_3x_{label}",
            (F.col(f"epss_{label}") > 0) & ((F.col("epss_current") / F.col(f"epss_{label}")) >= 3),
        )
        .withColumn(
            f"r_epss_02b_5x_{label}",
            (F.col(f"epss_{label}") > 0) & ((F.col("epss_current") / F.col(f"epss_{label}")) >= 5),
        )
        .withColumn(
            f"r_epss_02c_5x_delta_005_{label}",
            (F.col(f"epss_{label}") > 0)
            & ((F.col("epss_current") / F.col(f"epss_{label}")) >= 5)
            & ((F.col("epss_current") - F.col(f"epss_{label}")) >= 0.05),
        )
        .withColumn(
            f"r_epss_02d_10x_delta_010_{label}",
            (F.col(f"epss_{label}") > 0)
            & ((F.col("epss_current") / F.col(f"epss_{label}")) >= 10)
            & ((F.col("epss_current") - F.col(f"epss_{label}")) >= 0.10),
        )
        .withColumn(
            f"r_epss_02e_5x_delta_005_persistent_{label}",
            (F.col(f"epss_{label}") > 0)
            & ((F.col("epss_current") / F.col(f"epss_{label}")) >= 5)
            & ((F.col("epss_current") - F.col(f"epss_{label}")) >= 0.05)
            & (F.col("epss_d7") >= F.col(f"epss_{label}")),
        )
    )

rule_cols = [column for column in features_with_r_epss_02.columns if column.startswith("r_epss_02")]
total_cve_count = features_with_r_epss_02.select("cve_id").distinct().count()

summary_rows = []
for rule_col in rule_cols:
    hit_count = features_with_r_epss_02.filter(F.col(rule_col) == True).count()
    hit_rate_pct = round((hit_count / total_cve_count) * 100, 4)
    summary_rows.append((rule_col, hit_count, hit_rate_pct))

r_epss_02_summary = spark.createDataFrame(summary_rows, ["rule_id", "hit_count", "hit_rate_pct"])
r_epss_02_summary.createOrReplaceTempView("ds02_r_epss_02_summary")

display(r_epss_02_summary.orderBy(F.desc("hit_count")))

## 6. KEV 현재 상태 로드

In [ ]:
silver_kev = spark.table("silver_kev")

kev_current = (
    silver_kev
    .select(
        F.col("cve_id"),
        F.col("date_added").alias("kev_date_added"),
        F.col("known_ransomware_use"),
        F.col("required_action"),
    )
    .filter(F.col("cve_id").isNotNull())
    .filter(F.col("cve_id").startswith("CVE-"))
    .dropDuplicates(["cve_id"])
    .withColumn("is_kev", F.lit(True))
)

kev_current.createOrReplaceTempView("ds02_kev_current")

display(kev_current.limit(20))
print("silver_kev CVE count =", kev_current.count())

## 7. GHSA 최신 스냅샷과 분석 universe 구성

In [ ]:
ghsa_raw = (
    spark.read
    .json("Files/bronze/ghsa/*.jsonl")
    .withColumn("source_file", F.input_file_name())
    .withColumn(
        "snapshot_date",
        F.to_date(F.regexp_extract("source_file", r"ghsa_advisories_(\d{4}-\d{2}-\d{2})\.jsonl", 1)),
    )
)

latest_ghsa_date = ghsa_raw.agg(F.max("snapshot_date").alias("latest_date")).collect()[0]["latest_date"]
print("latest_ghsa_date =", latest_ghsa_date)

ghsa_current = ghsa_raw.filter(F.col("snapshot_date") == F.lit(latest_ghsa_date))

universe = (
    ghsa_current
    .filter(F.col("cve_id").isNotNull())
    .select("ghsa_id", "cve_id", "severity", "cvss", "vulnerabilities")
    .join(
        epss_features.select(
            "cve_id",
            "epss_current",
            "percentile_current",
            "epss_d30",
            "percentile_d30",
        ),
        on="cve_id",
        how="inner",
    )
    .join(kev_current.select("cve_id", "is_kev"), on="cve_id", how="left")
    .fillna({"is_kev": False})
)

universe.createOrReplaceTempView("ds02_evaluation_universe")

display(universe.limit(20))
print("evaluation universe rows =", universe.count())

## 8. 후보 신호와 proxy 컬럼 생성

In [ ]:
severity_rank = (
    F.when(F.lower(F.col("severity")) == "low", 1)
    .when(F.lower(F.col("severity")).isin("moderate", "medium"), 2)
    .when(F.lower(F.col("severity")) == "high", 3)
    .when(F.lower(F.col("severity")) == "critical", 4)
    .otherwise(0)
)

evaluation_table = (
    universe
    .withColumn("severity_rank", severity_rank)
    .withColumn("r_epss_01b_d30", (F.col("percentile_d30") < 0.90) & (F.col("percentile_current") >= 0.95))
    .withColumn("r_epss_01c_d30", (F.col("percentile_d30") < 0.90) & (F.col("percentile_current") >= 0.99))
    .withColumn(
        "r_epss_02c_d30",
        (F.col("epss_d30") > 0)
        & ((F.col("epss_current") / F.col("epss_d30")) >= 5)
        & ((F.col("epss_current") - F.col("epss_d30")) >= 0.05),
    )
    .withColumn(
        "r_epss_02d_d30",
        (F.col("epss_d30") > 0)
        & ((F.col("epss_current") / F.col("epss_d30")) >= 10)
        & ((F.col("epss_current") - F.col("epss_d30")) >= 0.10),
    )
    .withColumn("r_ghsa_current_cvss_gte_9", F.col("cvss.score").cast("double") >= 9.0)
    .withColumn("r_ghsa_current_cvss_gte_8", F.col("cvss.score").cast("double") >= 8.0)
    .withColumn("r_ghsa_current_critical", F.col("severity_rank") == 4)
    .withColumn("r_ghsa_current_high", F.col("severity_rank") == 3)
    .withColumn("proxy_kev", F.col("is_kev"))
    .withColumn("proxy_epss_top_5", F.col("percentile_current") >= 0.95)
    .withColumn("proxy_epss_top_1", F.col("percentile_current") >= 0.99)
    .withColumn("proxy_ghsa_critical", F.col("severity_rank") == 4)
    .withColumn("proxy_cvss_gte_9", F.col("cvss.score").cast("double") >= 9.0)
    .withColumn(
        "proxy_patched",
        F.coalesce(
            F.expr("""
                exists(vulnerabilities, v ->
                    v.first_patched_version is not null
                    and trim(v.first_patched_version) != ''
                    and lower(trim(v.first_patched_version)) != 'null'
                )
            """),
            F.lit(False),
        ),
    )
)

evaluation_table.createOrReplaceTempView("ds02_evaluation_table")

display(evaluation_table.limit(20))

## 9. Overlap / Precision-like / Lift 검증

In [ ]:
candidate_rules = [
    "r_epss_01b_d30",
    "r_epss_01c_d30",
    "r_epss_02c_d30",
    "r_epss_02d_d30",
    "r_ghsa_current_cvss_gte_9",
    "r_ghsa_current_cvss_gte_8",
    "r_ghsa_current_critical",
    "r_ghsa_current_high",
    "is_kev",
]

proxy_rules = [
    "proxy_kev",
    "proxy_epss_top_5",
    "proxy_epss_top_1",
    "proxy_ghsa_critical",
    "proxy_cvss_gte_9",
    "proxy_patched",
]

evaluation_cached = evaluation_table.select("cve_id", *candidate_rules, *proxy_rules).cache()
total_count = evaluation_cached.count()

agg_exprs = []
for candidate in candidate_rules:
    agg_exprs.append(F.sum(F.when(F.col(candidate) == True, 1).otherwise(0)).alias(f"{candidate}__hit"))

for proxy in proxy_rules:
    agg_exprs.append(F.sum(F.when(F.col(proxy) == True, 1).otherwise(0)).alias(f"{proxy}__base"))

for candidate in candidate_rules:
    for proxy in proxy_rules:
        agg_exprs.append(
            F.sum(F.when((F.col(candidate) == True) & (F.col(proxy) == True), 1).otherwise(0))
            .alias(f"{candidate}__{proxy}__overlap")
        )

agg_row = evaluation_cached.agg(*agg_exprs).collect()[0].asDict()

rows = []
for candidate in candidate_rules:
    candidate_count = agg_row[f"{candidate}__hit"]
    for proxy in proxy_rules:
        proxy_count = agg_row[f"{proxy}__base"]
        overlap_count = agg_row[f"{candidate}__{proxy}__overlap"]

        precision_like_pct = 0.0 if candidate_count == 0 else round(overlap_count / candidate_count * 100, 2)
        baseline_pct = 0.0 if total_count == 0 else round(proxy_count / total_count * 100, 2)
        lift = None
        if candidate_count > 0 and proxy_count > 0:
            lift = round((overlap_count / candidate_count) / (proxy_count / total_count), 2)

        rows.append((candidate, proxy, int(candidate_count), int(overlap_count), precision_like_pct, baseline_pct, lift))

overlap_summary = spark.createDataFrame(
    rows,
    ["candidate_rule", "proxy_rule", "candidate_hit_count", "overlap_count", "precision_like_pct", "baseline_pct", "lift"],
)

overlap_summary.createOrReplaceTempView("ds02_overlap_summary")

display(overlap_summary.orderBy("candidate_rule", F.desc("lift")))
display(
    overlap_summary
    .filter((F.col("proxy_rule") == "proxy_kev") & (F.col("candidate_rule") != "is_kev"))
    .select("candidate_rule", "candidate_hit_count", "overlap_count", "precision_like_pct", "baseline_pct", "lift")
    .orderBy(F.desc("lift"))
)

## 10. DS-02 후보 신호 temp view

In [ ]:
ds02_risk_change_signal_candidates = (
    epss_features
    .join(kev_current.select("cve_id", "kev_date_added", "is_kev"), on="cve_id", how="left")
    .fillna({"is_kev": False})
    .withColumn("r_epss_top5_entry_d30", (F.col("percentile_d30") < 0.90) & (F.col("percentile_current") >= 0.95))
    .withColumn(
        "r_epss_rise_d30",
        (F.col("epss_d30") > 0)
        & ((F.col("epss_current") / F.col("epss_d30")) >= 5)
        & ((F.col("epss_current") - F.col("epss_d30")) >= 0.05),
    )
    .withColumn("r_kev_current", F.col("is_kev"))
    .withColumn("r_kev_current_with_epss_top5", F.col("is_kev") & (F.col("percentile_current") >= 0.95))
    .withColumn("r_kev_current_with_epss_top1", F.col("is_kev") & (F.col("percentile_current") >= 0.99))
    .withColumn("r_kev_current_with_epss_rise_d30", F.col("is_kev") & F.col("r_epss_rise_d30"))
    .withColumn("r_kev_current_with_epss_top5_entry_d30", F.col("is_kev") & F.col("r_epss_top5_entry_d30"))
    .withColumn(
        "has_ds02_candidate_signal",
        F.col("r_epss_top5_entry_d30") | F.col("r_epss_rise_d30") | F.col("r_kev_current"),
    )
)

ds02_risk_change_signal_candidates.createOrReplaceTempView("ds02_risk_change_signal_candidates")

display(
    ds02_risk_change_signal_candidates
    .filter(F.col("has_ds02_candidate_signal") == True)
    .select(
        "cve_id",
        "epss_current",
        "percentile_current",
        "epss_d30",
        "percentile_d30",
        "kev_date_added",
        "r_epss_top5_entry_d30",
        "r_epss_rise_d30",
        "r_kev_current",
        "r_kev_current_with_epss_top5",
        "r_kev_current_with_epss_rise_d30",
        "r_kev_current_with_epss_top5_entry_d30",
    )
    .orderBy(F.desc("r_kev_current"), F.desc("percentile_current"), F.desc("epss_current"))
    .limit(100)
)

## 11. 완료 기준 확인

In [ ]:
required_temp_views = [
    "ds02_epss_features",
    "ds02_r_epss_01_summary",
    "ds02_r_epss_02_summary",
    "ds02_kev_current",
    "ds02_evaluation_universe",
    "ds02_evaluation_table",
    "ds02_overlap_summary",
    "ds02_risk_change_signal_candidates",
]

for view_name in required_temp_views:
    row_count = spark.table(view_name).count()
    print(f"{view_name}: {row_count}")

assert spark.table("ds02_r_epss_01_summary").filter(F.col("rule_id") == "r_epss_01b_top_5_entry_d30").count() == 1
assert spark.table("ds02_r_epss_02_summary").filter(F.col("rule_id") == "r_epss_02c_5x_delta_005_d30").count() == 1
assert spark.table("ds02_kev_current").filter(F.col("cve_id").isNull()).count() == 0

display(spark.sql("""
SELECT
    SUM(CASE WHEN r_epss_top5_entry_d30 THEN 1 ELSE 0 END) AS epss_top5_entry_d30_count,
    SUM(CASE WHEN r_epss_rise_d30 THEN 1 ELSE 0 END) AS epss_rise_d30_count,
    SUM(CASE WHEN r_kev_current THEN 1 ELSE 0 END) AS kev_current_count,
    SUM(CASE WHEN r_kev_current_with_epss_top5 THEN 1 ELSE 0 END) AS kev_with_epss_top5_count,
    SUM(CASE WHEN r_kev_current_with_epss_rise_d30 THEN 1 ELSE 0 END) AS kev_with_epss_rise_d30_count,
    SUM(CASE WHEN r_kev_current_with_epss_top5_entry_d30 THEN 1 ELSE 0 END) AS kev_with_epss_top5_entry_d30_count
FROM ds02_risk_change_signal_candidates
"""))